## The model training

### This Notebook aims to create a machine learning model and train it using the engineered data

In [63]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRFRegressor
from sklearn.preprocessing import LabelEncoder

### 3.1 Load the data

In [50]:
train_df = pd.read_csv('train_engineered.csv')
test_df = pd.read_csv('test_engineered.csv')

In [51]:
train_df.head()

,UniqueID,next_3m_txn_count,TransactionAmount_sum,TransactionAmount_mean,TransactionAmount_median,TransactionAmount_std,TransactionAmount_count,TransactionAmount_max,TransactionAmount_min,StatementBalance_mean,...,NetInterestIncome_mean,NetInterestIncome_std,NetInterestRevenue_sum,NetInterestRevenue_mean,Product_nunique,Gender,IncomeCategory,AnnualGrossIncome,OccupationCategory,CustomerStatus
0,00093e2d-9e1e-4061-ad27-a79b8ff9e165,129,-894563.40,-572.703841,-925.520,88833.409077,1562,1000000.00,-1000000.00,79175.340800,...,1304.165333,2414.125438,10647.86,236.619111,3.0,M,High Income,2244579.16,Management / Executive,Active Customer
1,0011d60f-a4e2-4333-81fc-2d557a82109b,16,-2000656.68,-7970.743745,-301.590,132902.058472,251,1000000.00,-1000000.00,463595.304900,...,-1775.770476,3450.277593,20482.99,975.380476,1.0,M,High Income,1251211.80,Management / Executive,Active Customer
2,0016f1e2-64c1-4c65-a668-1dc6bf3b5875,117,-127.16,-2.890000,-2.020,3.039344,44,0.00,-8.05,279.836136,...,-2.015238,0.428388,105.33,5.015714,1.0,M,Middle Income,344297.42,Other / Unclassified,Active Customer
3,001aa3c5-632d-435e-a421-cc3615ccef4d,70,87729.20,258.027059,-895.765,89647.208240,340,1000000.00,-698552.99,81728.208706,...,-433.280976,852.424778,6773.93,165.217805,2.0,F,Very High Income,1846455.79,Management / Executive,Active Customer
4,00298c6f-4f9d-4f28-b72c-ad0e56e9eb84,393,44110.45,20.650960,-143.645,9755.953589,2136,122286.62,-38055.67,155872.668043,...,-463.207273,859.880422,6383.57,290.162273,2.0,M,Very High Income,2176785.64,Management / Executive,Active Customer


In [52]:
print(train_df.shape)
print(test_df.shape)

(8360, 27)
(3584, 26)


In [53]:
categorical_cols = train_df.select_dtypes(include= 'str').columns

print(categorical_cols)

Index(['UniqueID', 'Gender', 'IncomeCategory', 'OccupationCategory',
       'CustomerStatus'],
      dtype='str')


### 3.2. Encoding the categorical into numerical to ensure the model processes correctly to these columns

In [ ]:
# numerical encode
for col in categorical_cols:
    if col != 'UniqueID':
        le = LabelEncoder()

        train_df[col] = le.fit_transform(
            train_df[col].astype(str)
        )

        test_df[col] = le.transform(
            test_df[col].astype(str)
        )

In [55]:
X = train_df.drop(
    ['UniqueID', 'next_3m_txn_count'],
    axis= 1
)

y = train_df['next_3m_txn_count']

In [56]:
y

0       129
1        16
2       117
3        70
4       393
       ... 
8355    366
8356    152
8357     13
8358     27
8359     55
Name: next_3m_txn_count, Length: 8360, dtype: int64

In [57]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 8360 entries, 0 to 8359
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   TransactionAmount_sum     8360 non-null   float64
 1   TransactionAmount_mean    8360 non-null   float64
 2   TransactionAmount_median  8360 non-null   float64
 3   TransactionAmount_std     8360 non-null   float64
 4   TransactionAmount_count   8360 non-null   int64  
 5   TransactionAmount_max     8360 non-null   float64
 6   TransactionAmount_min     8360 non-null   float64
 7   StatementBalance_mean     8360 non-null   float64
 8   StatementBalance_median   8360 non-null   float64
 9   StatementBalance_std      8360 non-null   float64
 10  StatementBalance_min      8360 non-null   float64
 11  StatementBalance_max      8360 non-null   float64
 12  Credit                    8360 non-null   int64  
 13  Debit                     8360 non-null   int64  
 14  NetInterestIncome_s

In [58]:
(X.dtypes == 'object').sum()

np.int64(0)

In [59]:
X.isna().sum().sum()

np.int64(0)

In [60]:
X

,TransactionAmount_sum,TransactionAmount_mean,TransactionAmount_median,TransactionAmount_std,TransactionAmount_count,TransactionAmount_max,TransactionAmount_min,StatementBalance_mean,StatementBalance_median,StatementBalance_std,...,NetInterestIncome_mean,NetInterestIncome_std,NetInterestRevenue_sum,NetInterestRevenue_mean,Product_nunique,Gender,IncomeCategory,AnnualGrossIncome,OccupationCategory,CustomerStatus
0,-894563.40,-572.703841,-925.520,88833.409077,1562,1000000.00,-1000000.00,79175.340800,39337.810,152572.842104,...,1304.165333,2414.125438,10647.86,236.619111,3.0,1,0,2244579.160,9,0
1,-2000656.68,-7970.743745,-301.590,132902.058472,251,1000000.00,-1000000.00,463595.304900,177155.140,436019.174486,...,-1775.770476,3450.277593,20482.99,975.380476,1.0,1,0,1251211.800,9,0
2,-127.16,-2.890000,-2.020,3.039344,44,0.00,-8.05,279.836136,287.955,48.508435,...,-2.015238,0.428388,105.33,5.015714,1.0,1,3,344297.420,12,0
3,87729.20,258.027059,-895.765,89647.208240,340,1000000.00,-698552.99,81728.208706,50704.050,138267.899794,...,-433.280976,852.424778,6773.93,165.217805,2.0,0,7,1846455.790,9,0
4,44110.45,20.650960,-143.645,9755.953589,2136,122286.62,-38055.67,155872.668043,155083.425,83783.388739,...,-463.207273,859.880422,6383.57,290.162273,2.0,1,7,2176785.640,9,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8355,-5002042.23,-1211.735036,-665.370,63528.663831,4128,1000000.00,-1000000.00,669549.585521,725509.515,300918.436626,...,11206.484250,27539.228200,63993.72,1599.843000,2.0,1,7,1564492.620,9,0
8356,7587.73,7.288886,-21.700,32151.061156,1041,365281.38,-761293.90,5785.300365,627.270,42253.456934,...,-20.128000,14.718194,19.78,3.956000,2.0,1,4,0.000,11,0
8357,-212.81,-1.564779,-19.990,81.314614,136,357.36,-125.26,138.223676,125.825,125.945006,...,-0.397273,0.351773,96.87,4.403182,1.0,1,1,1171.920,11,0
8358,1188.35,21.606364,-62.120,298.797792,55,1194.06,-485.44,997.038364,961.460,345.824313,...,-101.569655,74.686190,0.00,0.000000,2.0,0,5,632755.155,11,0


####

### 3.3. Splitting the data into train and evaluation sets

In [61]:
# split the data
X_tain, X_valid, y_train, y_valid = train_test_split(X, y, train_size= 0.8, random_state= 44)

####

###

#### 3.4. Model selection and training

In [64]:
# random forest regressor and xboost
models = {
    'RFRegressor': RandomForestRegressor(
    n_estimators= 200,
    random_state= 42,
    n_jobs= -1
), 
    'XGBoost': XGBRFRegressor(n_estimators= 500, learning_rate= 0.02)
}